<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px;">
<img src="../Modulo III/Figures/iteso.jpg" style="height:110px; background:white; padding:6px; border-radius:8px;"/>
<div style="border-left:2px solid white; padding-left:28px; color:white;">
<h1 style="font-size:1.5em;">Ingeniería y Ciencia de Datos</h1>
<h3>Laboratorio de Procesamiento de Datos</h3>
<p>Módulo III: Weight of Evidence e Information Value</p>
</div></div>

El **WOE** (*Weight of Evidence*) expresa la evidencia relativa que aporta un grupo de una variable explicativa respecto de una variable objetivo binaria. Compara cómo se distribuyen los casos con evento y sin evento entre los grupos de esa característica.

Se utiliza especialmente en **riesgo crediticio y scorecards con regresión logística**. También puede aplicarse a abandono de clientes, respuesta a campañas y ocurrencia de siniestros. Su propósito es transformar categorías o intervalos en valores numéricos basados en su relación con el objetivo.

**Aclaración:** WOE corresponde a cada grupo, no es por sí solo una medida global del poder predictivo de toda la variable. El e **IV** (*Information Value*), presentado más adelante, reúne las contribuciones de todos los grupos. **WOE** no es el coeficiente de una regresión ni una probabilidad.

Conviene entender WOE en términos de **eventos y no eventos**:

- **Evento, $Y=1$:** ocurrió el resultado definido como objetivo, por ejemplo incumplimiento, abandono o compra.
- **No evento, $Y=0$:** no ocurrió ese resultado.

Para un grupo $g$, distinguimos su **tasa de evento**, $N_{1g}/(N_{0g}+N_{1g})$, de su **distribución de eventos**, $N_{1g}/N_1$. La primera usa como denominador las personas del grupo, la segunda usa todos los eventos de la muestra. WOE utiliza distribuciones por clase.

Para el grupo $g$, definimos:

| Símbolo | Significado | Denominador correcto |
|---|---|---|
| $N_{0g}$ | No eventos del grupo | Conteo |
| $N_{1g}$ | Eventos del grupo | Conteo |
| $N_0$ | No eventos de toda la muestra | Suma de no eventos de todos los grupos |
| $N_1$ | Eventos de toda la muestra | Suma de eventos de todos los grupos |
| $d_{0g}=N_{0g}/N_0$ | Distribución de no eventos | Total de la clase 0 |
| $d_{1g}=N_{1g}/N_1$ | Distribución de eventos | Total de la clase 1 |

$$WOE_g=\ln\left(\frac{\text{distribución de no eventos del grupo}}{\text{distribución de eventos del grupo}}\right)=\ln\left(\frac{N_{0g}/N_0}{N_{1g}/N_1}\right) = \ln\left(\frac{d_{0g}}{d_{1g}}\right).$$


$N_{0g}$ y $N_{1g}$ son los conteos de cada clase en el grupo: $N_0$ y $N_1$, los totales de cada clase en toda la muestra. Las distribuciones suman uno por clase.

**Ejemplo:** si un grupo contiene 40 de los 200 no eventos y 10 de los 100 eventos, WOE = ln(0.20/0.10) = ln(2) ≈ 0.6931.

| WOE | Interpretación bajo esta convención |
|---|---|
| Positivo | Mayor presencia relativa entre no eventos |
| Negativo | Mayor presencia relativa entre eventos |
| Cero | Distribuciones iguales para ese grupo |

Cero no significa probabilidad de evento cero.

#### ¿Por qué un logaritmo?

El cociente puede ir de cero a infinito; su logaritmo expresa la evidencia alrededor de cero. Cocientes recíprocos reciben valores opuestos: ln(2) = 0.693 y ln(1/2) = -0.693. Si el cociente es 1, WOE es cero. El logaritmo conecta además con las log-odds de la regresión logística.


#### Ejemplo: 700 solicitudes históricas

La tabla siguiente es un ejemplo construido para entender las operaciones, no datos de ListenData ni de clientes reales. Tiene 600 no eventos y 100 eventos. Primero observamos conteos; después calculamos distribuciones, WOE e IV sin redondear los valores intermedios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

manual = pd.DataFrame({
    'no_eventos': [300, 200, 100],
    'eventos': [20, 30, 50]
}, index=pd.Index(['bueno', 'regular', 'malo'], name='historial'))
manual['personas'] = manual['no_eventos'] + manual['eventos']
manual['tasa_evento'] = manual['eventos'] / manual['personas']
manual

In [ ]:
total_no_eventos = manual['no_eventos'].sum()
total_eventos = manual['eventos'].sum()
tasa_global = total_eventos / manual['personas'].sum()

print('Total no eventos:', total_no_eventos)
print('Total eventos:', total_eventos)
print(f'Tasa global de incumplimiento: {tasa_global:.2%}')

In [ ]:
# Calculo d0, d1 , cocientge y WOE


Para historial **bueno**:
1. Hay 20 incumplimientos entre 320 personas: tasa del grupo = 20/320 = **6.25%**.
2. El grupo contiene 300 de los 600 no eventos: $d_0=50\%$.
3. Contiene 20 de los 100 eventos: $d_1=20\%$.
4. El cociente 0.50/0.20 = 2.5 indica presencia relativa 2.5 veces mayor entre no eventos.
5. Su WOE es ln(2.5) = **0.9163**.

**La diferencia clave:** $d_1$ responde «¿dónde están los eventos?», la tasa del grupo responde «¿qué ocurre dentro de este grupo?». Las distribuciones d0 y d1 suman uno **por columna**, las tasas de grupos no tienen que sumar uno.

En historial **malo**, la tasa es 33.33%, superior a la global de 14.29%, su WOE es negativo. En **regular**, la tasa es ligeramente menor que la global, su WOE es positivo pero cercano a cero.

## Pasos para calcular WOE

Para una **variable continua**:

1. Dividir los datos en diez grupos, o menos según su distribución y tamaño de muestra.
3. Contar los eventos y no eventos de cada grupo o *bin*.
4. Dividir cada conteo entre el total de su clase para obtener las distribuciones de eventos y no eventos.
5. Calcular el logaritmo natural del cociente distribución de no eventos/distribución de eventos.

Para una **variable categórica**, sus categorías pueden ser los grupos iniciales: se omite el primer paso y se comienza con los conteos.

**Ejemplo de una fila de la tabla:** 40 no eventos, 10 eventos; totales de clase 200 y 100. Las distribuciones son 0.20 y 0.10, WOE ≈ 0.6931 y tasa de evento = 10/50 = 0.20. Aunque algunos valores coincidan aquí, sus denominadores y significados son diferentes.


## Usos de WOE

WOE permite representar una variable explicativa mediante grupos o intervalos cuyo comportamiento se describe a partir de la distribución del objetivo: conteos de eventos y no eventos. Después se reemplaza cada grupo por su valor WOE.

Conviene separar dos operaciones: **binning** construye los grupos, **WOE** calcula la evidencia de cada grupo. Un **binning por cuantiles** usa la distribución de X (características de entrada, predictores) para obtener tamaños similares. No garantiza por sí solo que los grupos tengan una relación monótona con Y (variable objetivo).

| Aplicación | Evento | Uso de la transformación |
|---|---|---|
| Crédito | Incumplimiento | Representar rangos de ingreso o deuda |
| Retención | Abandono | Representar segmentos de uso del servicio |
| Campañas | Respuesta o compra | Representar categorías de contacto |
| Seguros | Ocurrencia de siniestro | Representar categorías y rangos de riesgo |

La transformación es **supervisada**: requiere el objetivo histórico. Una asociación observada no demuestra causalidad ni garantiza mejor rendimiento que otras codificaciones.

### Para variables explicativas continuas

Primero se crean intervalos o grupos de la variable continua. Después pueden combinarse intervalos vecinos con WOE similar y reemplazarse los grupos por sus valores WOE. El modelo recibe esos valores en lugar de las mediciones originales.

**Ejemplo ilustrativo:** un ingreso de 18,000 y otro de 19,500 podrían pertenecer al mismo intervalo y recibir ambos WOE = 0.35. Eso no significa que los ingresos sean iguales, la representación agrupada deja de distinguirlos.

Crear muchos grupos iniciales se conoce como *fine classing*, fusionarlos en grupos más amplios, como *coarse classing*. Al fusionar se suman conteos y se recalculan distribuciones y WOE: no se promedian directamente los WOE antiguos.


### Para variables explicativas categóricas

Pueden combinarse categorías con WOE semejante y reemplazarse las categorías resultantes por sus valores WOE. En lugar de etiquetas originales, el modelo recibe una columna numérica.

**Ejemplo:** un mapa aprendido podría ser `{'correo': 0.4, 'SMS': 0.1, 'teléfono': -0.5}`. Cada fila recibe el valor de su categoría. 

**Matiz:** aunque la columna resultante sea numérica, contiene un conjunto discreto de valores, uno por grupo. No se vuelve una medición continua en el sentido estadístico ni impone distancias físicas entre categorías. WOE puede evitar varias dummies, pero pierde información frente a representaciones más flexibles.


Las categorías con WOE similar tienen cocientes de distribución de eventos/no eventos parecidos. En ese sentido muestran un comportamiento semejante respecto del objetivo y pueden ser candidatas a agruparse.


## Recomendaciones relacionadas con WOE

1. Procurar que cada categoría o intervalo contenga al menos el 5% de las observaciones.
2. Procurar conteos distintos de cero tanto para eventos como para no eventos.
3. Revisar que los WOE de los grupos aporten diferencias relevantes y agrupar categorías muy similares cuando corresponda.
4. Buscar un WOE monótono, creciente o decreciente, cuando el comportamiento esperado de la variable y el diseño del scorecard lo justifiquen.
5. Tratar los valores faltantes como un grupo separado.


## Número de intervalos o grupos

Como punto de partida se utilizan a menudo diez o veinte intervalos. Idealmente, cada grupo contiene suficiente muestra, la referencia propone al menos 5% de los casos. Menos intervalos producen una representación más suave y simple, más intervalos permiten capturar más detalle, pero también ruido.

El número apropiado depende de muestra, conteos por clase, valores repetidos y conocimiento del dominio. No elegir veinte automáticamente: si no hay suficientes valores distintos, algunos bordes se repiten y puede haber menos bins efectivos.


## Manejo de grupos con cero eventos o cero no eventos

Si un grupo no contiene eventos o no eventos, el cociente puede producir WOE infinito. Una alternativa descrita en la referencia añade **0.5** a los conteos de ambas clases del grupo:

$$WOE_{g,ajustado}=\ln\left(\frac{(N_{0g}+0.5)/N_0}{(N_{1g}+0.5)/N_1}\right).$$

Aquí el logaritmo se aplica al cociente completo. Si solo se toma el logaritmo del numerador y luego se divide, se obtiene otra expresión incorrecta.

$$\widetilde d_{0g}=\frac{N_{0g}+\alpha}{N_0+G\alpha},\qquad\widetilde d_{1g}=\frac{N_{1g}+\alpha}{N_1+G\alpha},\qquad WOE_g=\ln(\widetilde d_{0g}/\widetilde d_{1g}).$$

Así cada distribución suma uno. Esta variante puede dar resultados ligeramente diferentes de la primera expresión: siempre declarar la utilizada.


## Beneficios de WOE

1. **Puede reducir el efecto de valores extremos sobre la representación.** Un salario muy alto puede compartir intervalo con otros ingresos elevados y recibir el mismo WOE. Esto no detecta ni corrige por sí solo errores de captura; se pierde detalle dentro del bin.
2. **Permite representar faltantes de forma explícita.** Un grupo separado puede revelar si la ausencia de información está asociada con el objetivo, siempre que exista suficiente evidencia histórica.
3. **Puede evitar crear varias variables dummy.** Una variable categórica puede representarse mediante una columna WOE. La reducción de columnas no garantiza mejor desempeño que one-hot.

**Balance práctico:** WOE puede facilitar inspeccionar grupos y representar relaciones no lineales mediante intervalos. Sus costos son pérdida de detalle, dependencia del objetivo y riesgo de sobreajuste. Para usarlo en registros nuevos deben conservarse mapas, bordes, política de faltantes y versión del modelo.

En crédito, seguros o campañas, un modelo útil requiere además definir horizonte, disponibilidad de variables, costos de errores y evaluación de sesgos. WOE no demuestra causalidad ni garantiza decisiones justas.

## Valor de información (IV)

El **valor de información (IV)** permite explorar y ordenar variables según la separación que sus grupos presentan entre las dos clases del objetivo. Es una herramienta habitual de selección preliminar en scorecards.

**Diferencia esencial:** WOE es un valor por grupo, **IV** es un número para toda la variable, calculado para una muestra y una partición concretas. IV no es una columna que se asigne a cada persona ni un coeficiente del modelo.

IV mide separación **univariada**, no importancia conjunta. Una variable de IV alto puede ser redundante con otra o contener información posterior al evento. Una de IV bajo puede intervenir en una interacción útil. El ranking ayuda a explorar, pero no garantiza el mejor subconjunto predictivo.

$$IV(X)=\sum_g\left(d_{0g}-d_{1g}\right)WOE_g=\sum_g\left(d_{0g}-d_{1g}\right)\ln\left(\frac{d_{0g}}{d_{1g}}\right).$$

Cada grupo aporta $IV_g=(d_{0g}-d_{1g})WOE_g$. El IV de la variable es **la suma**, no el promedio, de esas contribuciones.

**Ejemplo:** d0=0.20, d1=0.10 y WOE=ln(2) dan contribución IV = 0.10 × 0.6931 ≈ 0.0693. Para obtener el IV total faltan las contribuciones de los demás grupos.

Con distribuciones positivas, cada contribución es no negativa porque la diferencia y el logaritmo tienen el mismo signo. Si ambas distribuciones coinciden en todos los grupos, IV=0. IV no es un porcentaje, no es información mutua y **no está limitado a 1**.

Si se invierte el cociente de WOE, debe invertirse también la diferencia a d1-d0: así se conserva IV.

| Valor de información | Separación predictiva orientativa |
|---|---|
| Menor que 0.02 | Insignificante |
| De 0.02 a menos de 0.10 | Débil |
| De 0.10 a menos de 0.30 | Moderada |
| De 0.30 a 0.50 | Fuerte |
| Mayor que 0.50 | Sospechosamente alta: revisar |

Estos rangos son convenciones de scorecards, no umbrales universales ni pruebas estadísticas.

Interpretación práctica del IV:

1. **Menor que 0.02:** la separación univariada es insignificante; revisar si la variable aporta por otras vías antes de descartarla.
2. **Entre 0.02 y 0.10:** relación débil con las distribuciones de las clases.
3. **Entre 0.10 y 0.30:** separación moderada; puede ser candidata para el modelo.
4. **Entre 0.30 y 0.50:** separación fuerte; revisar estabilidad y redundancia.
5. **Mayor que 0.50:** revisar especialmente fuga, grupos pequeños, identificadores y momento de disponibilidad. También puede existir separación real.


In [ ]:
#Del ejemplo inicial
manual['IV_grupo'] = (manual['d0'] - manual['d1']) * manual['WOE']
iv_manual = manual['IV_grupo'].sum()

In [ ]:
manual

In [ ]:
def mostrar_figura(nombre):
    plt.tight_layout()
    #plt.savefig(figures / f'{nombre}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    
fig, ejes = plt.subplots(1, 3, figsize=(13, 4))
manual[['d0', 'd1']].plot.bar(ax=ejes[0], color=['#167d9a', '#ce7742'], rot=0)
ejes[0].set(title='Distribuciones por clase', ylabel='Proporción dentro de cada clase', xlabel='Historial')
manual['WOE'].plot.bar(ax=ejes[1], color=['#167d9a', '#6b9d70', '#ce7742'], rot=0)
ejes[1].axhline(0, color='black', linewidth=1)
ejes[1].set(title='WOE por grupo', ylabel='Logaritmo del cociente', xlabel='Historial')
manual['IV_grupo'].plot.bar(ax=ejes[2], color='#6b9d70', rot=0)
ejes[2].set(title=f'Contribuciones: IV total = {iv_manual:.4f}', ylabel='IV del grupo', xlabel='Historial')
mostrar_figura('01_ejemplo_manual')

mapa_manual = manual['WOE'].to_dict()
solicitudes = pd.DataFrame({'historial': ['bueno', 'malo', 'regular', 'bueno']})
solicitudes['historial_WOE'] = solicitudes['historial'].map(mapa_manual)
display(solicitudes)
assert solicitudes.loc[0, 'historial_WOE'] == solicitudes.loc[3, 'historial_WOE']

El primer gráfico compara d0 y d1, el segundo muestra el signo de WOE, el tercero muestra las contribuciones al IV. Un grupo con WOE negativo también aporta **positivamente** al IV: lo relevante para la separación es la diferencia entre distribuciones, no una dirección preferida.

Tras aprender la tabla histórica, crear una variable WOE es una operación de **mapeo**. Si dos personas pertenecen al mismo grupo, ambas reciben el mismo valor. Los valores no se vuelven a estimar para cada persona. Un WOE para cada fila no significa un modelo personalizado por fila.


## Ejemplo práctico

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

#función de ayuda para calcular woe
def calcular_woe_iv(datos, objetivo, bins=10, alpha=0.5):
    respuesta = datos[objetivo]
    if respuesta.isna().any() or set(respuesta.unique()) != {0, 1}:
        raise ValueError('El objetivo debe contener ambas clases 0 y 1, sin faltantes.')
    if alpha <= 0 or bins < 2:
        raise ValueError('Se requieren alpha > 0 y al menos dos bins.')
    tablas, resumen = [], []
    for variable in datos.columns.drop(objetivo):
        serie = datos[variable]
        es_numerica = pd.api.types.is_numeric_dtype(serie)
        if es_numerica and serie.nunique() > bins:
            grupos = pd.qcut(serie, q=bins, duplicates='drop').astype('string')
        else:
            grupos = serie.astype('string')
        grupos = grupos.fillna('__FALTANTE__') # para valores faltantes
        conteos = pd.crosstab(grupos, respuesta).reindex(columns=[0, 1], fill_value=0)
        conteos.columns = ['no_eventos', 'eventos']
        numero_grupos = len(conteos)
        # total de eventos
        conteos['n'] = conteos['no_eventos'] + conteos['eventos']
        # tasa de eventos
        conteos['tasa_evento'] = conteos['eventos'] / conteos['n']

        # Calculo de WOE, alpha por si no hay eventos en grupos
        conteos['dist_no_eventos'] = (conteos['no_eventos'] + alpha) / ((respuesta == 0).sum() + numero_grupos * alpha)
        conteos['dist_eventos'] = (conteos['eventos'] + alpha) / ((respuesta == 1).sum() + numero_grupos * alpha)
        
        conteos['WOE'] = np.log(conteos['dist_no_eventos'] / conteos['dist_eventos'])
        conteos['IV_grupo'] = (conteos['dist_no_eventos'] - conteos['dist_eventos']) * conteos['WOE']
        resumen.append({'variable': variable, 'IV': conteos['IV_grupo'].sum(), 'grupos': numero_grupos})
        
        conteos.index.name = 'grupo'
        conteos = conteos.reset_index()
        conteos.insert(0, 'variable', variable)
        tablas.append(conteos)
    return pd.DataFrame(resumen).sort_values('IV', ascending=False).reset_index(drop=True), pd.concat(tablas, ignore_index=True)


In [ ]:
import os
# Usando binary_admisions.csc
ruta_datos = os.path.join('Data', 'binary_admisiones.csv')
mydata = pd.read_csv(ruta_datos)
mydata['rank'] = mydata['rank'].astype('string')
display(mydata.head())
print('Evento: admit=1, admisión, no evento: admit=0, no admisión.')
print('Número de observaciones:', len(mydata))

In [ ]:
#calcular_woe_iv


In [ ]:
#Gráficas de IV y WOE

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
iv.set_index('variable')['IV'].sort_values().plot.barh(ax=ejes[0], color='#167d9a')
ejes[0].set(title='IV: separación por variable', xlabel='Information Value', ylabel='Variable')
tabla_rank = woe.loc[woe['variable'] == 'rank'].set_index('grupo')
tabla_rank['WOE'].plot.bar(ax=ejes[1], color='#c87842', rot=0)
ejes[1].axhline(0, color='black', linewidth=1)
ejes[1].set(title='WOE: evidencia por categoría de rank', xlabel='Categoría', ylabel='WOE')
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
mapa_rank = tabla_rank['WOE'].to_dict()
ejemplo_transformado = mydata[['rank', 'admit']].head(8).copy()
ejemplo_transformado['rank_WOE'] = ejemplo_transformado['rank'].map(mapa_rank)
display(ejemplo_transformado)
print('Estos WOE se calculan con datos históricos; no usar admit de registros nuevos para transformarlos.')

In [ ]:
comparacion_bins = []
for cantidad in [3, 5, 10, 20]:
    resumen, tablas = calcular_woe_iv(mydata, objetivo='admit', bins=cantidad)
    for variable in ['gre', 'gpa']:
        fila = resumen.set_index('variable').loc[variable]
        comparacion_bins.append({'bins_solicitados': cantidad, 'variable': variable,
                                 'bins_obtenidos': int(fila['grupos']), 'IV': fila['IV']})
comparacion_bins = pd.DataFrame(comparacion_bins)
display(comparacion_bins)
comparacion_bins.pivot(index='bins_solicitados', columns='variable', values='IV').plot(marker='o')
plt.title('IV descriptivo bajo distintas agrupaciones')
plt.xlabel('Número de bins solicitados')
plt.ylabel('Information Value')
plt.tight_layout()
plt.show()
plt.close()

El ranking IV compara variables, mientras que la tabla WOE describe grupos. En el cálculo manual, el grupo A tiene WOE ≈ 0.6931 e IV de grupo ≈ 0.0693, el IV total es ≈ 0.0811. La tasa del grupo A es 20%, un concepto diferente de su WOE.

En admisiones, los WOE negativos de `rank` indican mayor presencia relativa entre admitidos bajo la convención elegida. No prueban que cambiar de institución cause una admisión. La comparación de bins demuestra que el IV depende de la representación y no es una propiedad fija de la variable.

1. **¿Qué denominadores usa WOE?** Los totales de cada clase, no el total del grupo.
2. **¿WOE cero implica probabilidad cero?** No: las distribuciones de clases coinciden para ese grupo; con conteos sin suavizar, su tasa es la global.
3. **¿Por qué una contribución IV puede ser positiva con WOE negativo?** La diferencia d0-d1 también es negativa; su producto es positivo.
4. **¿Cómo se recalcula WOE al unir grupos?** Sumando conteos, recalculando distribuciones y tomando el logaritmo del nuevo cociente.
5. **¿Qué se aprende solo con entrenamiento?** Bordes, agrupaciones, conteos, mapas WOE y cualquier filtro por IV. En validación cruzada se reaprenden dentro de cada fold.
6. **¿Es obligatorio elegir el mayor IV?** No. Revisar estabilidad, fuga, rareza, redundancia y desempeño conjunto fuera de ajuste.


### ¿Para qué sirven WOE/IV y en qué aplicaciones tienen sentido?

| Aplicación | Evento posible | Uso de WOE | Papel de IV |
|---|---|---|---|
| Riesgo crediticio | Incumplimiento en doce meses | Representar ingreso, deuda e historial para un scorecard | Explorar candidatos antes de modelar |
| Retención de clientes | Abandono en el siguiente trimestre | Codificar patrones de uso conocidos antes del abandono | Comparar separación univariada |
| Respuesta a campañas | Compra o respuesta después del contacto | Representar segmentos con distinto patrón de respuesta | Priorizar variables para estudiar, no determinar efecto causal |
| Seguros | Ocurrencia de siniestro en un horizonte definido | Codificar rangos y categorías de riesgo | Explorar asociación con ocurrencia, no monto de pérdida |
| Calidad industrial | Pieza defectuosa | Agrupar medidas disponibles antes de inspección final | Explorar variables; no omitir interacciones relevantes |

Su uso clásico es la **regresión logística binaria y los scorecards**: convierten variables heterogéneas en evidencia relativa y permiten revisar tablas de grupos. Pueden usarse como representaciones para otros clasificadores, pero ni WOE ni la selección por IV son óptimos para todos ellos. Árboles y ensembles pueden explotar relaciones e interacciones que el binning o un filtro univariado perderían.

### Qué ganamos y qué no

- **Posible ventaja:** menos columnas que one-hot, tratamiento explícito de faltantes y representación no lineal de una variable original mediante intervalos.
- **Costo:** pérdida de detalle dentro de bins, dependencia de etiquetas, riesgo de sobreajuste y necesidad de guardar reglas para producción.
- **No garantizan:** precisión mayor, buena calibración, causalidad, ausencia de sesgo ni decisiones justas o rentables.

Hasta aquí tal vez tengamos algunas preguntas:

**¿WOE es un peso del modelo?** No. Es la codificación de un grupo. El coeficiente de regresión se aprende después y multiplica ese valor.

**¿IV se asigna a cada persona?** No. Cada variable tiene un IV calculado para esa muestra y esa partición. Las personas reciben los WOE de sus grupos.

**¿Un IV bajo implica que la variable no sirve?** No necesariamente: puede participar en interacciones o aportar algo que otra variable no contiene. La evaluación conjunta decide.

**¿WOE debe ser monótono?** No por definición. Es una preferencia frecuente de scorecards cuando corresponde al dominio. Forzar monotonía puede esconder una relación en U. La regresión exige linealidad en las log-odds respecto de sus predictores, no monotonía obligatoria respecto de una variable original antes de transformarla.

**¿Qué pasa si hay muy pocos eventos?** Las distribuciones siguen siendo calculables si ambas clases existen, pero las estimaciones pueden ser inestables. Revisar conteos, fusionar cuando tenga sentido y validar; el suavizado no reemplaza muestra.

**¿Se puede usar un objetivo continuo o multiclase?** Las fórmulas de esta guía son para un objetivo binario. Existen extensiones, pero no se obtienen aplicando sin cambios el mismo cálculo.

# Práctica de Laboratorio: Abandono de clientes Telco mediante WOE e IV

Explorar cómo los grupos de características se asocian con abandono y permanencia, interpretar IV y formular hipótesis de retención. Un cliente que abandona deja de utilizar el servicio en el periodo registrado; detectar asociación no demuestra que una característica cause el abandono.

**Datos:** Telco Customer Churn, 7,043 clientes y características de contratación, servicios, cargos y demografía. La fuente del caso es [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), descargaremos una copia distribuida en el repositorio de IBM y la conservaremos en `Data`.

**Sobre este dataset:** `abandono=1` si `Churn='Yes'`, `abandono=0` si permanece. Aquí:
- WOE = ln(distribución de no eventos/distribución de eventos). Por tanto, WOE positivo significa mayor presencia relativa entre quienes permanecen
- negativo, entre quienes abandonan. 

### 1. Vista general y carga de datos

Ejecutar previamente la celda que define `calcular_woe_iv`. La descarga se realiza solo si no hay copia local. `TotalCharges` contiene espacios que deben tratarse como faltantes, no como cargos de cero.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ruta_telco = os.path.join('Data', 'Telco-Customer-Churn.csv')
telco_original = pd.read_csv(ruta_telco, na_values=[' '])
print('Dataset cargado de:', ruta_telco)

columnas_necesarias = {'customerID', 'Churn', 'SeniorCitizen', 'TotalCharges', 'InternetService', 'Contract', 'tenure'}
print('Clientes:', len(telco_original), '| Columnas:', telco_original.shape[1])

In [ ]:
telco_original.info()

In [ ]:
telco_original.isna().sum()

### 2. Preparación de las variables y separación de los dos análisis

Normalizamos el inicio de los nombres de columnas, convertimos `totalCharges` a número y eliminamos `customerID` de las variables explicativas: un identificador único puede inflar IV sin generalizar. `seniorCitizen` se representa como categoría, no como cantidad continua.

Los cargos totales faltantes se mantienen en un grupo explícito. Revisaremos si corresponden a clientes con antigüedad cero, sin asignarles artificialmente cargos cero.

Como en el caso de referencia, analizaremos:
- **Todos los clientes:** variables generales, sin los seis servicios que solo se ofrecen con Internet.
- **Clientes con Internet:** seguridad, respaldo, protección de dispositivos, soporte y streaming.

`No internet service` no significa lo mismo que `No`: puede indicar que el servicio adicional no aplica. Filtrar la segunda población evita confundir esa situación con rechazar un complemento.

El objetivo es descriptivo: no estamos midiendo qué variables causan el abandono ni validando un clasificador. Los denominadores de las distribuciones cambian en el análisis restringido, por lo que no se deben mezclar directamente los IV de ambas poblaciones.

In [ ]:
telco = telco_original.rename(columns={columna: columna[0].lower() + columna[1:] for columna in telco_original.columns}).copy()
telco['abandono'] = telco['churn'].map({'Yes': 1, 'No': 0}).astype(int)
telco['seniorCitizen'] = telco['seniorCitizen'].map({0: 'No', 1: 'Yes'}).astype('string')
telco['totalCharges'] = pd.to_numeric(telco['totalCharges'], errors='coerce')
telco = telco.drop(columns=['customerID', 'churn'])
telco.head()

In [ ]:
servicios_internet = ['onlineSecurity', 'techSupport', 'onlineBackup', 'streamingTV', 'streamingMovies', 'deviceProtection']
telco_general = telco.drop(columns=servicios_internet)
telco_internet = telco.loc[telco['internetService'] != 'No', servicios_internet + ['abandono']].copy()

In [ ]:
print('Cargos totales faltantes:', telco['totalCharges'].isna().sum())
display(telco.loc[telco['totalCharges'].isna(), ['tenure', 'monthlyCharges', 'totalCharges', 'abandono']])
display(pd.DataFrame({
    'clientes': [len(telco_general), len(telco_internet)],
    'abandono': [telco_general['abandono'].mean(), telco_internet['abandono'].mean()]
}, index=['Todos los clientes', 'Solo clientes con Internet']))


### 3. Análisis de IV y extremos de WOE: todos los clientes

Calculamos WOE e IV para las trece variables generales y clasificamos el IV con los rangos orientativos de la parte teórica. Los valores superiores a 0.50 se marcan para revisión: pueden reflejar separación real, agrupaciones pequeñas, información temporal o problemas de datos, no necesariamente un error.

El gráfico de IV compara variables. El de extremos muestra el WOE mínimo y máximo de sus grupos: un extremo grande apoyado por muy pocos clientes no debe interpretarse como evidencia sólida. La tabla incluye el número de grupos y una alerta de grupos con menos de 5% de la población analizada.

Contrato, antigüedad y tipo de Internet son candidatos a examinar. Las relaciones que se observan se interpretan con el contexto: un contrato de mayor duración puede implicar menos oportunidades de cancelar, pero el análisis transversal no permite atribuirle un efecto causal.

Reutilizamos `calcular_woe_iv` con diez cuantiles y alpha=0.5.

In [ ]:
def interpretar_iv(valor):
    if valor < 0.02:
        return 'Insignificante'
    if valor < 0.10:
        return 'Débil'
    if valor < 0.30:
        return 'Moderada'
    if valor <= 0.50:
        return 'Fuerte'
    return 'Alta: revisar'

In [ ]:
iv_telco_general, woe_telco_general = calcular_woe_iv(telco_general, objetivo='abandono', bins=10, alpha=0.5)
iv_telco_general['interpretacion'] = iv_telco_general['IV'].map(interpretar_iv)
grupos_pequenos = woe_telco_general.assign(pequeno=woe_telco_general['n'] / len(telco_general) < 0.05).groupby('variable')['pequeno'].sum()
iv_telco_general['grupos_menores_5pct'] = iv_telco_general['variable'].map(grupos_pequenos)
iv_telco_general


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 6))
iv_telco_general.set_index('variable')['IV'].sort_values().plot.barh(ax=ejes[0], color='#167d9a')
ejes[0].axvline(0.5, color='#c87842', linestyle='--', label='IV > 0.50: revisar')
ejes[0].set(title='IV: variables generales', xlabel='Information Value', ylabel='Variable')
ejes[0].legend()
extremos_generales = woe_telco_general.groupby('variable')['WOE'].agg(minimo='min', maximo='max')
extremos_generales.plot.barh(ax=ejes[1], color=['#c87842', '#6c9b73'])
ejes[1].axvline(0, color='black', linewidth=1)
ejes[1].set(title='Extremos de WOE por variable', xlabel='WOE', ylabel='Variable')
plt.tight_layout()
plt.show()
plt.close()


### 4. IV y extremos de WOE: clientes con Internet

Repetimos el cálculo para los seis complementos dentro de la población que sí tiene Internet. Comparamos el WOE de contratar y no contratar cada complemento respecto de abandono/permanencia en esa población.

Una asociación entre soporte o seguridad y menor abandono puede reflejar diferencias en contratos, antigüedad o preferencias. No demuestra que ofrecer el complemento a cualquier cliente reduzca su abandono. Esta es una hipótesis de negocio que necesita validación y, si se busca causalidad, un diseño adecuado.

La tabla y las gráficas siguientes reproducen el tipo de análisis del caso de referencia con las fórmulas y funciones de esta guía. No se mezclan los clientes sin Internet ni se comparan los IV como si ambas poblaciones fueran idénticas.

In [ ]:
iv_telco_internet, woe_telco_internet = calcular_woe_iv(telco_internet, objetivo='abandono', bins=10, alpha=0.5)
iv_telco_internet['interpretacion'] = iv_telco_internet['IV'].map(interpretar_iv)
iv_telco_internet


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
iv_telco_internet.set_index('variable')['IV'].sort_values().plot.barh(ax=ejes[0], color='#6c9b73')
ejes[0].set(title='IV: complementos de Internet', xlabel='Information Value', ylabel='Variable')
extremos_internet = woe_telco_internet.groupby('variable')['WOE'].agg(minimo='min', maximo='max')
extremos_internet.plot.barh(ax=ejes[1], color=['#c87842', '#167d9a'])
ejes[1].axvline(0, color='black', linewidth=1)
ejes[1].set(title='Extremos de WOE: solo clientes con Internet', xlabel='WOE', ylabel='Variable')
plt.tight_layout()
plt.show()
plt.close()

woe_telco_internet.loc[woe_telco_internet['variable'].isin(['onlineSecurity', 'techSupport'])].round(4)

### 5. Análisis detallado de WOE

Examinamos seis características generales y tres complementos destacados. Las barras comparan evidencia relativa; las tablas anteriores permiten revisar tasas y tamaños.

- **Antigüedad y cargos totales:** suelen estar relacionados. `totalCharges` acumula facturación; `tenure × monthlyCharges` es una aproximación, no una identidad exacta para todos los clientes, porque las tarifas pueden cambiar. Un IV alto en ambos no garantiza información adicional independiente.
- **Contrato:** comparar mensual, anual y bienal. WOE positivo implica mayor presencia entre quienes permanecen bajo nuestra convención; no determina por sí solo una probabilidad individual.
- **Tipo de Internet:** comparar DSL, fibra y ausencia de servicio. Una diferencia observada puede motivar investigar calidad, precios o composición de clientes; no demuestra una falla técnica.
- **Forma de pago:** distinguir medios automáticos y no automáticos, sin asumir que obligar a cambiar de medio reducirá cancelaciones.
- **Complementos de Internet:** estudiar seguridad, soporte y respaldo dentro de la población elegible. Su asociación puede estar ligada a planes, antigüedad u otras variables.

Los intervalos numéricos se grafican en su orden de magnitud, no en orden alfabético. El grupo de faltantes aparece por separado. Las categorías nominales no adquieren un orden causal por aparecer en una gráfica.

In [ ]:
def tabla_telco_ordenada(variable, tablas, poblacion):
    tabla = tablas.loc[tablas['variable'] == variable].copy()
    serie = poblacion[variable]
    if pd.api.types.is_numeric_dtype(serie) and serie.nunique() > 10:
        orden = pd.qcut(serie, q=10, duplicates='drop').cat.categories.astype(str).tolist()
    elif variable == 'contract':
        orden = ['Month-to-month', 'One year', 'Two year']
    else:
        orden = sorted(tabla.loc[tabla['grupo'] != '__FALTANTE__', 'grupo'].astype(str).tolist())
    orden = orden + ['__FALTANTE__']
    tabla['grupo'] = pd.Categorical(tabla['grupo'], categories=orden, ordered=True)
    return tabla.sort_values('grupo')

variables_detalle = ['contract', 'tenure', 'internetService', 'paymentMethod', 'totalCharges', 'monthlyCharges']
nombres_detalle = {'contract': 'Contrato', 'tenure': 'Antigüedad en meses', 'internetService': 'Tipo de Internet',
                  'paymentMethod': 'Forma de pago', 'totalCharges': 'Cargos totales', 'monthlyCharges': 'Cargos mensuales'}
fig, ejes = plt.subplots(2, 3, figsize=(16, 10))
for eje, variable in zip(ejes.flat, variables_detalle):
    tabla = tabla_telco_ordenada(variable, woe_telco_general, telco_general)
    eje.bar(np.arange(len(tabla)), tabla['WOE'], color=np.where(tabla['WOE'] >= 0, '#167d9a', '#c87842'))
    eje.set_xticks(np.arange(len(tabla)), tabla['grupo'].astype('string'), rotation=55, ha='right', fontsize=8)
    eje.axhline(0, color='black', linewidth=1)
    eje.set(title=nombres_detalle[variable], ylabel='WOE')
fig.suptitle('Clientes Telco: WOE positivo = mayor presencia relativa entre quienes permanecen')
plt.tight_layout()
plt.show()
plt.close()

fig, ejes = plt.subplots(1, 3, figsize=(12, 4))
for eje, variable in zip(ejes, ['onlineSecurity', 'techSupport', 'onlineBackup']):
    tabla = tabla_telco_ordenada(variable, woe_telco_internet, telco_internet)
    eje.bar(tabla['grupo'].astype('string'), tabla['WOE'], color=np.where(tabla['WOE'] >= 0, '#167d9a', '#c87842'))
    eje.axhline(0, color='black', linewidth=1)
    eje.set(title=variable, ylabel='WOE')
plt.tight_layout()
plt.show()
plt.close()

contrato_woe = woe_telco_general.loc[woe_telco_general['variable'] == 'contract'].set_index('grupo')['WOE']
assert contrato_woe['Month-to-month'] < 0 < contrato_woe['Two year']
print('La dirección de WOE es coherente con abandono=1.')

### 6. Describir segmentos de clientes que abandonan

Para convertir las tablas en preguntas de negocio, revisamos antigüedad corta, contrato mensual, cargos mensuales elevados, fibra, pago electrónico y ausencia de seguridad/soporte. Los puntos de corte siguientes son ejemplos de segmentación para comparar con el caso de referencia, **no umbrales optimizados ni una regla de clasificación validada**.

Calcularemos cuántos clientes hay en cada segmento y su tasa histórica de abandono, además de una combinación de condiciones. Los segmentos individuales se superponen: sus conteos no deben sumarse como si fueran categorías excluyentes.

Una tasa mayor que la global describe esta muestra; no es la probabilidad individual futura de cada miembro. Una combinación con pocos clientes puede ser muy inestable. Para construir un perfil predictivo se necesitaría separar muestras, evaluar generalización y revisar calibración y costos.

Comparar estos resultados con las tablas WOE: un segmento más frecuente entre abandonos puede tener WOE negativo, pero su tasa usa un denominador distinto de la distribución de eventos.

In [ ]:
segmentos_telco = {
    'Antigüedad menor de 3 meses': telco['tenure'] < 3,
    'Contrato mensual': telco['contract'] == 'Month-to-month',
    'Cargos mensuales mayores que 50': telco['monthlyCharges'] > 50,
    'Internet de fibra óptica': telco['internetService'] == 'Fiber optic',
    'Pago por cheque electrónico': telco['paymentMethod'] == 'Electronic check',
    'Con Internet, sin seguridad': (telco['internetService'] != 'No') & (telco['onlineSecurity'] == 'No'),
    'Con Internet, sin soporte': (telco['internetService'] != 'No') & (telco['techSupport'] == 'No')
}
perfil_combinado = (segmentos_telco['Antigüedad menor de 3 meses'] & segmentos_telco['Contrato mensual']
                    & segmentos_telco['Cargos mensuales mayores que 50'] & segmentos_telco['Internet de fibra óptica']
                    & segmentos_telco['Pago por cheque electrónico'] & segmentos_telco['Con Internet, sin seguridad']
                    & segmentos_telco['Con Internet, sin soporte'])
segmentos_telco['Combinación ilustrativa de condiciones'] = perfil_combinado
filas_segmentos = []
for nombre, mascara in segmentos_telco.items():
    muestra = telco.loc[mascara, 'abandono']
    filas_segmentos.append({'segmento': nombre, 'clientes': len(muestra), 'abandonos': muestra.sum(),
                           'tasa_abandono': muestra.mean() if len(muestra) else np.nan})
resumen_segmentos = pd.DataFrame(filas_segmentos)
resumen_segmentos['tasa_global'] = telco['abandono'].mean()
resumen_segmentos

### 7. De los hallazgos a acciones que se puedan evaluar

Compartir resultados con atención al cliente, producto y marketing para formular hipótesis, no para convertir asociaciones en obligaciones automáticas.

| Hipótesis de negocio | Intervención posible | Comprobación necesaria |
|---|---|---|
| Dificultades al iniciar el servicio | Acompañamiento voluntario durante los primeros meses | Comparar retención y satisfacción con un grupo de control |
| Problemas de facturación o pagos | Recordatorios consentidos y opciones de pago claras | Medir fallos de pago, quejas y abandono |
| Diferencias entre planes o tecnologías | Auditar precios y calidad del servicio | Separar calidad de composición de clientes |
| Clientes con necesidades de soporte | Mejorar acceso y tiempos de atención | Evaluar experiencia, uso y resultados de retención |
| Promociones de permanencia | Ofertas transparentes y voluntarias | Medir beneficio neto, no solo cancelaciones |

No justificar restricciones de pago ni trato desigual únicamente con un IV alto. Para evaluar efectos, proponer un experimento con asignación aleatoria cuando sea viable, periodo de seguimiento definido y métricas de satisfacción, retención, costos y rentabilidad.

In [ ]:
import os
carpeta_practica = os.path.join('Data', 'Practica_Telco')
os.makedirs(carpeta_practica, exist_ok=True)
iv_telco_general.to_csv(os.path.join(carpeta_practica, 'IV_todos_clientes.csv'), index=False)
woe_telco_general.to_csv(os.path.join(carpeta_practica, 'WOE_todos_clientes.csv'), index=False)
iv_telco_internet.to_csv(os.path.join(carpeta_practica, 'IV_clientes_internet.csv'), index=False)
woe_telco_internet.to_csv(os.path.join(carpeta_practica, 'WOE_clientes_internet.csv'), index=False)
resumen_segmentos.to_csv(os.path.join(carpeta_practica, 'segmentos_abandono.csv'), index=False)


### 8. Preguntas de laboratorio

1. Explicar por qué se excluye `customerID` y por qué `seniorCitizen` se trata como categoría.
2. Usar los conteos de `contract='Month-to-month'` para calcular a mano d0, d1, WOE e IV de grupo, con y sin alpha=0.5. Distinguir tasa del grupo de distribución de eventos.
3. Identificar las tres variables generales de mayor IV. ¿Un IV > 0.50 prueba fuga? ¿Qué revisión temporal y de muestra harías?
4. Comparar soporte y seguridad dentro de clientes con Internet. ¿Qué confusión introduce incluir `No internet service` como si fuera simplemente `No`?
5. Explicar por qué los IV de dos poblaciones distintas no se pueden combinar directamente en un mismo ranking.
6. Comparar la tasa de abandono de los segmentos con la global. Reportar cuántos casos sostienen cada cifra y por qué no es una predicción individual validada.
7. Repetir los cálculos con cinco bins y comparar los rankings. No elegir el mejor número solo por el IV de esta muestra.
8. Proponer una acción de retención, su grupo de comparación, horizonte, métricas y posibles factores de confusión.
